# Treinamento QLoRA 
Este notebook contém a exploração, a geração dos dados de treino, o treino do LoRA e as **avaliações iniciais** (células 04 e 06). Os erros reais do relatório vieram destas saídas.

> A avaliação final (busca com recall, placar das 3 configurações) está em `02_pipeline_e_placar.ipynb`, que usa um prompt único para treino e avaliação (veja RELATORIO.md, erro #1).


In [8]:
!pip install -r requirements.txt

(saída do pip omitida — dependências em requirements.txt)


### 01 - Conhecendo o Qwen2.5-3B-Instruct em 4-bits (Focado em Extração JSON)


Carrega o modelo otimizado, mostra a anatomia de um LLM
de 3 bilhões de parâmetros e como ele prevê os próximos tokens.

In [4]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELO = "Qwen/Qwen2.5-3B-Instruct"

# Escolhe o melhor hardware disponível
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Hardware: {device}")

# Configuração para espremer o modelo de 3B para caber nos 8GB de VRAM da RTX 4060
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# %% 1) Carregar tokenizador e modelo quantizado
print("\nCarregando modelo em 4-bits (pode demorar na primeira vez)...")
tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(
    MODELO,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

# %% 2) Anatomia: quantos parâmetros e como estão organizados
total = sum(p.numel() for p in model.parameters())
cfg = model.config
print(f"\nParâmetros totais (antes da quantização): {total:,}".replace(",", "."))
print(f"Camadas (blocos transformer): {cfg.num_hidden_layers}")
print(f"Dimensão do embedding:        {cfg.hidden_size}")
print(f"Cabeças de atenção:           {cfg.num_attention_heads}")
print(f"Tamanho do vocabulário:       {cfg.vocab_size}")

# %% 3) Texto -> tokens -> texto (Foco no JSON esperado)
frase = '{"local": "praça da sé", "crime": "assalto", "arma": true}'
ids = tokenizer(frase)["input_ids"]
print(f"\nFrase: {frase!r}")
print(f"{len(ids)} tokens: {ids}")
print("Pedaços:", [tokenizer.decode([i]) for i in ids])

# %% 4) O que o modelo "acha" que vem depois? (top-5 próximos tokens)
prompt = "<|im_start|>user\nExtraia as entidades: 'Roubo na avenida paulista.'<|im_end|>\n<|im_start|>assistant\n{\"local\":"
entrada = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():
    logits = model(**entrada).logits[0, -1]          # scores do próximo token
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 5)
print(f"\nPrompt: {prompt!r}")
for p, i in zip(top.values, top.indices):
    print(f"  {tokenizer.decode([int(i)])!r:>12}  {p.item():.1%}")

# %% 5) Geração: repetir a previsão várias vezes
prompt_teste = "<|im_start|>user\nExtraia as entidades em JSON do relato: 'Acabaram de roubar um carro na avenida paulista, o cara estava com um oitão.'<|im_end|>\n<|im_start|>assistant\n"
entrada_teste = tokenizer(prompt_teste, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():
    saida = model.generate(**entrada_teste, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)

print("\n--- Geração do modelo base Instruct (Zero-Shot) ---")
print(tokenizer.decode(saida[0][entrada_teste["input_ids"].shape[1]:], skip_special_tokens=True))

Hardware: cuda

Carregando modelo em 4-bits (pode demorar na primeira vez)...


Loading weights: 100%|██████████| 434/434 [00:08<00:00, 52.53it/s]



Parâmetros totais (antes da quantização): 1.698.672.640
Camadas (blocos transformer): 36
Dimensão do embedding:        2048
Cabeças de atenção:           16
Tamanho do vocabulário:       151936

Frase: '{"local": "praça da sé", "crime": "assalto", "arma": true}'
21 tokens: [4913, 2438, 788, 330, 31659, 17472, 2994, 18744, 497, 330, 84354, 788, 330, 395, 75677, 497, 330, 19726, 788, 830, 92]
Pedaços: ['{"', 'local', '":', ' "', 'pra', 'ça', ' da', ' sé', '",', ' "', 'crime', '":', ' "', 'ass', 'alto', '",', ' "', 'arma', '":', ' true', '}']

Prompt: '<|im_start|>user\nExtraia as entidades: \'Roubo na avenida paulista.\'<|im_end|>\n<|im_start|>assistant\n{"local":'
          ' "'  100.0%
         ' ["'  0.0%
         ' {"'  0.0%
        ' "\''  0.0%
        ' "\\'  0.0%

--- Geração do modelo base Instruct (Zero-Shot) ---
Aqui está a extração das entidades em formato JSON do relato fornecido:

{
  "event": "roubo",
  "location": {
    "street": "Avenida Paulista",
    "city": ""
  },
  

### 02 - Gerando arquivos jsonl de treino e teste para treinar modelo com LoRA

In [ ]:
import json
import random
from pathlib import Path

random.seed(42)

# MATRIZ DE DADOS (Vocabulário Controlado e Gírias)

LOCAIS = ["praça da sé", "metro consolação", "rua augusta", "ponto de ônibus",
          "avenida paulista", "minha casa", "estacionamento do shopping", 
          "rua de trás", "centro da cidade", "beco escuro", "passarela"]

CRIMES = ["roubo", "assalto", "furto", "homicídio", "latrocínio", 
          "ameaça", "sequestro", "vandalismo", "agressão"]

# Dicionários de mapeamento: Gíria/Texto -> Categoria Oficial
ARMAS_FOGO = {"um 38": "fogo", "um oitão": "fogo", "uma pistola": "fogo", 
              "arma de fogo": "fogo", "um revólver": "fogo", "um cano": "fogo", "arma":"fogo", "deu um tiro": "fogo"}
ARMAS_BRANCA = {"uma faca": "branca", "um facão": "branca", "uma peixeira": "branca", 
                "um pedaço de pau": "branca", "um canivete": "branca"}
DESARMADO = {"na mão grande": "desarmado", "só no grito": "desarmado", 
             "desarmado": "desarmado", "limpo": "desarmado", "sem arma": "desarmado", "no soco": "desarmado"}
ARMA_DESC = {"não vi se tinha arma": "desconhecido", "estava escuro": "desconhecido"}

VITIMA_LESAO = {"tomei um soco": "lesao", "me cortou": "lesao", "tá sangrando": "lesao", "foi esfaqueado": "lesao"}
VITIMA_FATAL = {"mataram ele": "fatalidade", "tá morto": "fatalidade", "veio a óbito": "fatalidade"}
VITIMA_ILESA = {"só o susto mesmo": "sem_lesao", "ngm se machucou": "sem_lesao", "tá todo mundo bem": "sem_lesao"}
VITIMA_DESC = {"não vi a vítima": "desconhecido", "ninguém viu depois": "desconhecido"}

# Dados que só vão aparecer no teste (para testar generalização)
LOCAIS_INEDITOS = ["estação da luz", "mercadão", "padaria da esquina", "posto de gasolina"]
CRIMES_INEDITOS = ["roubo de celular", "arrastão"]
ARMAS_INEDITAS = {"fuzil": "fogo", "três oitão": "fogo", "um gargalo de garrafa": "branca", "estilete": "branca", "na força": "desarmado", "do nada": "desconhecido"}
VITIMAS_INEDITAS = {"levou um tiro na perna": "lesao", "assassinado no local": "fatalidade", "saiu intacto": "sem_lesao", "não sei como a pessoa tá": "desconhecido"}

ESTRUTURAS_FRASES = [
    "Tava na {local} e rolou um {crime}. O bandido tava com {arma}. {vitima}.",
    "Acabaram de relatar {crime} aqui em {local}. Eles vieram {arma}, {vitima}.",
    "Mano, {crime} na {local}! Ameaçaram usando {arma}. {vitima}.",
    "Socorro, {crime} agora mesmo! Foi perto de {local}. O cara usou {arma}. {vitima}.",
    "Foi um {crime} horrível. {vitima}. Aconteceu em {local}. A arma era {arma}."
]

def gerar_exemplo(inedito=False):
    # Escolhe a base de dados dependendo se é inédito ou não
    lista_locais = LOCAIS + LOCAIS_INEDITOS if inedito else LOCAIS
    lista_crimes = CRIMES + CRIMES_INEDITOS if inedito else CRIMES
    
    dict_armas = {**ARMAS_FOGO, **ARMAS_BRANCA, **DESARMADO, **ARMA_DESC}
    if inedito: dict_armas.update(ARMAS_INEDITAS)
    
    dict_vitimas = {**VITIMA_LESAO, **VITIMA_FATAL, **VITIMA_ILESA, **VITIMA_DESC}
    if inedito: dict_vitimas.update(VITIMAS_INEDITAS)

    # 1. Sorteia as entidades
    local_real = random.choice(lista_locais)
    crime_real = random.choice(lista_crimes)
    arma_texto, arma_cat = random.choice(list(dict_armas.items()))
    vitima_texto, vitima_cat = random.choice(list(dict_vitimas.items()))

    # 15% de chance do cidadão não informar o local (Cenário caótico)
    local_label = local_real
    if random.random() < 0.15:
        local_real = "um lugar por aí"
        local_label = "desconhecido"

    # Mapeamento do Booleano 'arma' com base na categoria
    if arma_cat in ["fogo", "branca"]:
        arma_bool = True
    elif arma_cat == "desarmado":
        arma_bool = False
    else:
        arma_bool = None # Desconhecido vira null no JSON

    # 2. Constrói o texto usando um template aleatório
    texto = random.choice(ESTRUTURAS_FRASES).format(
        local=local_real, crime=crime_real, arma=arma_texto, vitima=vitima_texto
    )
    
    # Pequenos ajustes gramaticais para soar natural
    texto = texto.replace("com não vi", "e não vi").replace("usou desarmado", "veio desarmado")
    texto = texto.replace("usando na mão grande", "na mão grande").replace("com desarmado", "desarmado")

    # Sorteia um ID de B.O. fictício
    id_bo = f"BO-{random.randint(1000, 9999)}"
    
    # 20% das vezes, simula que a busca não achou nada (Para treinar a RECUSA)
    if random.random() < 0.20:
        tarefa = f"### Contexto Recuperado:\nNenhum documento relevante encontrado.\n### Tarefa:\nExtraia as entidades do contexto em formato JSON e cite a fonte. Se o contexto disser 'Nenhum documento relevante', retorne {{\"resultado\": \"não consta\"}}."
        codigo = '{"resultado": "não consta"}'
    else:
        # Gera o prompt normal com o contexto
        tarefa = f"### Contexto Recuperado (Fonte: {id_bo}):\n{texto}\n### Tarefa:\nExtraia as entidades do contexto em formato JSON e cite a fonte. Se o contexto disser 'Nenhum documento relevante', retorne {{\"resultado\": \"não consta\"}}."
        
        # 3. Constrói o JSON estrito (gabarito)
        gabarito_dict = {
            "local": local_label,
            "crime": crime_real.replace(" ", "_"),
            "arma": arma_bool,
            "tipo_arma": arma_cat,
            "estado_vitima": vitima_cat,
            "fonte": id_bo
        }
        codigo = json.dumps(gabarito_dict, ensure_ascii=False)
    
    # Retorna as duas chaves necessárias para o treino
    return {
        "tarefa": tarefa,
        "codigo": codigo,
        "tipo": "inedita" if inedito else "variacao"
    }

# GERAÇÃO E SALVAMENTO

treino, teste = [], []

# Gera 200 exemplos de treino apenas com o vocabulário base
for _ in range(200):
    treino.append(gerar_exemplo(inedito=False))

# Gera 30 exemplos de teste (15 variações conhecidas + 15 inéditas/desconhecidas)
for _ in range(15):
    teste.append(gerar_exemplo(inedito=False))
for _ in range(15):
    teste.append(gerar_exemplo(inedito=True))

random.shuffle(treino)
random.shuffle(teste)

Path("dados").mkdir(exist_ok=True)
for nome, dados in [("treino", treino), ("teste", teste)]:
    with open(f"dados/{nome}.jsonl", "w", encoding="utf-8") as arq:
        for ex in dados:
            # O ensure_ascii=False garante que acentos fiquem legíveis no JSONL
            arq.write(json.dumps(ex, ensure_ascii=False) + "\n")

n_var = sum(e["tipo"] == "variacao" for e in teste)
print(f"Treino: {len(treino)} exemplos")
print(f"Teste:  {len(teste)} exemplos ({n_var} variações + {len(teste) - n_var} inéditas)")
print("\nExemplo de treino:\n")
print(f"### Tarefa:\n{treino[0]['tarefa']}\n### Código:\n{treino[0]['codigo']}")

Treino: 200 exemplos
Teste:  30 exemplos (15 variações + 15 inéditas)

Exemplo de treino:

### Tarefa:
### Contexto Recuperado (Fonte: BO-3286):
Acabaram de relatar assalto aqui em passarela. Eles vieram um 38, ninguém viu depois.
### Tarefa:
Extraia as entidades do contexto em formato JSON e cite a fonte. Se o contexto disser 'Nenhum documento relevante', retorne {"resultado": "não consta"}.
### Código:
{"local": "passarela", "crime": "assalto", "arma": true, "tipo_arma": "fogo", "estado_vitima": "desconhecido", "fonte": "BO-3286"}


### 03 - Avaliando o Modelo Base

Avalia a capacidade de extração e formatação do modelo sem
o adaptador LoRA, rodando localmente (4-bits).

In [ ]:

import json
import os
import re
import sys
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# CONFIGURAÇÕES
USAR_MODELO_TREINADO = False          
MODELO_BASE = "Qwen/Qwen2.5-3B-Instruct"
ADAPTADOR_LORA = "modelo-lora-rag"
MAX_TOKENS = 150                      

try:
    os.chdir(Path(__file__).resolve().parent)
except NameError:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"

# CARREGAMENTO DO MODELO (Otimizado para 8GB VRAM) 
print(f"Hardware: {device}")
print("Configurando Quantização 4-bits...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Carregando Tokenizador e Modelo Base ({MODELO_BASE})...")
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
model = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=bnb_config,
    device_map="auto"
)

if USAR_MODELO_TREINADO:
    print(f"Injetando o Adaptador LoRA ({ADAPTADOR_LORA})...")
    model = PeftModel.from_pretrained(model, ADAPTADOR_LORA)

model.eval()

# CARREGAR DADOS DE TESTE 
try:
    teste = [json.loads(l) for l in open("dados/teste.jsonl", encoding="utf-8")]
except FileNotFoundError:
    print("Erro: Arquivo dados/teste.jsonl não encontrado. Rode o 02_dados_rag.py primeiro.")
    sys.exit(1)

print(f"\nAvaliando {'MODELO TREINADO (LoRA)' if USAR_MODELO_TREINADO else 'MODELO BASE'} | {len(teste)} tarefas\n")

# FUNÇÕES 
def gerar_saida(tarefa):
    """Monta o prompt no formato ChatML do Qwen e gera a resposta."""
    
    # O Qwen-3B-Instruct já é inteligente o suficiente para não precisar de 1-Shot,
    # um bom System Prompt já o força a gerar JSON.
    prompt = f"""<|im_start|>system
Você é um extrator de entidades JSON. Responda APENAS com o JSON.<|im_end|>
<|im_start|>user
{tarefa}<|im_end|>
<|im_start|>assistant
"""
    entrada = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        saida = model.generate(
            **entrada, 
            max_new_tokens=MAX_TOKENS, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    tamanho_prompt = entrada["input_ids"].shape[1]
    novos = saida[0][tamanho_prompt:]
    
    return tokenizer.decode(novos, skip_special_tokens=True).strip()

def extrair_json_seguro(texto):
    """Remove qualquer tag Markdown ou 'tagarelice' em volta do JSON."""
    match = re.search(r'\{.*?\}', texto, re.DOTALL)
    if match:
        return match.group(0)
    return texto 

def testa_extracao(gerado, esperado_str):
    texto_limpo = extrair_json_seguro(gerado)
    
    if not texto_limpo:
        return False, "Falha: O modelo não gerou um JSON."
    
    try:
        resultado = json.loads(texto_limpo)
        gabarito = json.loads(esperado_str)
        erros = []
        
        if gabarito.get("resultado") == "não consta":
            if resultado.get("resultado") != "não consta":
                return False, f"Falha: Esperava recusa ('não consta'), gerou {resultado}"
            return True, ""
            
        chaves_obrigatorias = ["local", "crime", "arma", "tipo_arma", "estado_vitima", "fonte"]
        for chave in chaves_obrigatorias:
            val_gerado = resultado.get(chave)
            val_esperado = gabarito.get(chave)
            if val_gerado != val_esperado:
                erros.append(f"[{chave}: esp={val_esperado!r} | ger={val_gerado!r}]")
                
        if erros:
            return False, " | ".join(erros)
            
        return True, ""
    except json.JSONDecodeError:
        return False, "Erro crítico: JSON quebrado."

# AVALIAÇÃO 
resultados = []
inicio = time.time()
for i, ex in enumerate(teste, 1):
    texto_gerado = gerar_saida(ex["tarefa"])
    ok, erro = testa_extracao(texto_gerado, ex["codigo"])
    
    json_resumo = extrair_json_seguro(texto_gerado)
    resumo_tela = json_resumo[:55].replace('\n', '') + ("..." if len(json_resumo) > 55 else "")
    
    resultados.append({**ex, "gerado": texto_gerado, "passou": ok, "erro": erro})
    print(f"[{i:2d}/{len(teste)}] {'PASSOU' if ok else 'falhou'} "
          f"gerou: {resumo_tela!r}  {erro}")

# RESUMO
print(f"\nTempo: {time.time() - inicio:.0f} s")
print("=" * 50)
for tipo in ["variacao", "inedita"]:
    grupo = [r for r in resultados if r.get("tipo") == tipo]
    if grupo:
        acertos = sum(r["passou"] for r in grupo)
        print(f"{tipo:<10}: {acertos:2d}/{len(grupo)}  ({acertos / len(grupo):.0%})")
total = sum(r["passou"] for r in resultados)
print(f"{'TOTAL':<10}: {total:2d}/{len(resultados)}  ({total / len(resultados):.0%})")
print("=" * 50)

# Mostra 2 exemplos para debug
for r in resultados[:2]:
    print(f"\nTAREFA: {r['tarefa']}")
    print(f"--- esperado ---\n{r['codigo']}")
    print(f"--- gerado ({'passou' if r['passou'] else 'falhou'}) ---\n{r['gerado']}")

# SALVAR RESULTADOS
Path("resultados").mkdir(exist_ok=True)
nome = "lora-treinado" if USAR_MODELO_TREINADO else "qwen-base"
with open(f"resultados/{nome}.json", "w", encoding="utf-8") as arq:
    json.dump(resultados, arq, ensure_ascii=False, indent=2)
print(f"\nResultados salvos em resultados/{nome}.json")

Hardware: cuda
Configurando Quantização 4-bits...
Carregando Tokenizador e Modelo Base (Qwen/Qwen2.5-3B-Instruct)...


Loading weights: 100%|██████████| 434/434 [00:06<00:00, 68.32it/s]



Avaliando MODELO BASE | 30 tarefas

[ 1/30] falhou gerou: '{"resultado": "não consta", "fonte": "BO-4893"}'  [local: esp='rua augusta' | ger=None] | [crime: esp='agressão' | ger=None] | [tipo_arma: esp='desconhecido' | ger=None] | [estado_vitima: esp='sem_lesao' | ger=None]
[ 2/30] falhou gerou: '{"resultado": "não consta", "fonte": "BO-ml424"}'  [local: esp='estacionamento do shopping' | ger=None] | [crime: esp='ameaça' | ger=None] | [tipo_arma: esp='desconhecido' | ger=None] | [estado_vitima: esp='sem_lesao' | ger=None] | [fonte: esp='BO-1424' | ger='BO-ml424']
[ 3/30] falhou gerou: '{"resultado": "não consta", "fonte": "BO-3ml5"}'  [local: esp='passarela' | ger=None] | [crime: esp='vandalismo' | ger=None] | [arma: esp=True | ger=None] | [tipo_arma: esp='branca' | ger=None] | [estado_vitima: esp='desconhecido' | ger=None] | [fonte: esp='BO-3255' | ger='BO-3ml5']
[ 4/30] falhou gerou: '{"resultado": "BO-2229", "entidades": [{"tipo": "Local"...'  Erro crítico: JSON quebrado.
[ 5/30] f

### 04 - Treinamento QLoRA (Qwen2.5-3B-Instruct)

Treina um adaptador LoRA leve em cima do modelo quantizado em 4-bits.

In [ ]:

import json
import random
import time
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODELO = "Qwen/Qwen2.5-3B-Instruct"
SAIDA = "modelo-lora-rag"
EPOCAS = 3
LOTE = 4  
LR = 5e-5 
MAX_LEN = 512

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Configurando Quantização 4-bits...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 
)

tokenizer = AutoTokenizer.from_pretrained(MODELO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Carregando modelo base...")
model = AutoModelForCausalLM.from_pretrained(
    MODELO,
    quantization_config=bnb_config,
    device_map={"":0}
)


model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

config_lora = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config_lora)
model.print_trainable_parameters()

treino = [json.loads(l) for l in open("dados/treino.jsonl", encoding="utf-8")]
teste = [json.loads(l) for l in open("dados/teste.jsonl", encoding="utf-8")]

def preparar_lote(exemplos):
    ids_lote, labels_lote = [], []
    for ex in exemplos:
        # Formato ChatML obrigatório do Qwen
        prompt = f"<|im_start|>system\nVocê é um extrator de entidades JSON.<|im_end|>\n<|im_start|>user\n{ex['tarefa']}<|im_end|>\n<|im_start|>assistant\n"
        resposta = f"{ex['codigo']}<|im_end|>"
        
        ids_p = tokenizer(prompt)["input_ids"]
        ids_r = tokenizer(resposta, add_special_tokens=False)["input_ids"]
        
        ids = (ids_p + ids_r)[:MAX_LEN]
        labels = ([-100] * len(ids_p) + ids_r)[:MAX_LEN]
        
        ids_lote.append(ids)
        labels_lote.append(labels)

    tam = max(len(x) for x in ids_lote)
    input_ids = [x + [tokenizer.pad_token_id] * (tam - len(x)) for x in ids_lote]
    mascara = [[1] * len(x) + [0] * (tam - len(x)) for x in ids_lote]
    labels = [x + [-100] * (tam - len(x)) for x in labels_lote]
    
    return {
        "input_ids": torch.tensor(input_ids, device=device),
        "attention_mask": torch.tensor(mascara, device=device),
        "labels": torch.tensor(labels, device=device),
    }

def loss_no_teste():
    model.eval()
    total = 0.0
    with torch.no_grad():
        for i in range(0, len(teste), LOTE):
            total += model(**preparar_lote(teste[i:i + LOTE])).loss.item()
    model.train()
    return total / ((len(teste) + LOTE - 1) // LOTE)

# Otimizador foca APENAS nos parâmetros do LoRA (requires_grad)
otimizador = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

print("\nIniciando Treinamento QLoRA...")
model.train()
passo = 0
hist_loss = []
inicio = time.time()

for epoca in range(1, EPOCAS + 1):
    random.shuffle(treino)
    for i in range(0, len(treino), LOTE):
        lote = preparar_lote(treino[i:i + LOTE])
        loss = model(**lote).loss
        loss.backward()
        otimizador.step()
        otimizador.zero_grad()

        passo += 1
        if passo % 10 == 0:
            print(f"Época {epoca} | Passo {passo} | Loss {loss.item():.3f}")
            hist_loss.append(loss.item())

    val = loss_no_teste()
    print(f"--> Fim da época {epoca}: Loss no Teste = {val:.3f}\n")

# Salva apenas o adaptador (alguns MBs, em vez de 6GB)
model.save_pretrained(SAIDA)
tokenizer.save_pretrained(SAIDA)
print(f"Treino concluído. Adaptador LoRA salvo em {SAIDA}/")

C:\Users\Victor T\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 16:44:09.319000 16924 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Configurando Quantização 4-bits...
Carregando modelo base...


Loading weights: 100%|██████████| 434/434 [00:04<00:00, 87.06it/s] 
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383

Iniciando Treinamento QLoRA...
Época 1 | Passo 10 | Loss 1.504
Época 1 | Passo 20 | Loss 0.882
Época 1 | Passo 30 | Loss 0.371
Época 1 | Passo 40 | Loss 0.219
Época 1 | Passo 50 | Loss 0.185
--> Fim da época 1: Loss no Teste = 0.158

Época 2 | Passo 60 | Loss 0.092
Época 2 | Passo 70 | Loss 0.000
Época 2 | Passo 80 | Loss 0.020
Época 2 | Passo 90 | Loss 0.069
Época 2 | Passo 100 | Loss 0.024
--> Fim da época 2: Loss no Teste = 0.039

Época 3 | Passo 110 | Loss 0.005
Época 3 | Passo 120 | Loss 0.001
Época 3 | Passo 130 | Loss 0.003
Época 3 | Passo 140 | Loss 0.009
Época 3 | Passo 150 | Loss 0.004
--> Fim da época 3: Loss no Teste = 0.022



Treino concluído. Adaptador LoRA salvo em modelo-lora-rag/


### 05 - Avaliando o modelo trenado com QLoRA

Avalia a capacidade de extração e formatação do modelo com
o adaptador LoRA, rodando localmente (4-bits).

In [ ]:

import json
import os
import re
import sys
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- CONFIGURAÇÕES ---
USAR_MODELO_TREINADO = True          
MODELO_BASE = "Qwen/Qwen2.5-3B-Instruct"
ADAPTADOR_LORA = "modelo-lora-rag"
MAX_TOKENS = 150                      

try:
    os.chdir(Path(__file__).resolve().parent)
except NameError:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- CARREGAMENTO DO MODELO (Otimizado para 8GB VRAM) ---
print(f"Hardware: {device}")
print("Configurando Quantização 4-bits...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Carregando Tokenizador e Modelo Base ({MODELO_BASE})...")
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
model = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=bnb_config,
    device_map="auto"
)

if USAR_MODELO_TREINADO:
    print(f"Injetando o Adaptador LoRA ({ADAPTADOR_LORA})...")
    model = PeftModel.from_pretrained(model, ADAPTADOR_LORA)

model.eval()

# CARREGAR DADOS DE TESTE
try:
    teste = [json.loads(l) for l in open("dados/teste.jsonl", encoding="utf-8")]
except FileNotFoundError:
    print("Erro: Arquivo dados/teste.jsonl não encontrado. Rode o 02_dados_rag.py primeiro.")
    sys.exit(1)

print(f"\nAvaliando {'MODELO TREINADO (LoRA)' if USAR_MODELO_TREINADO else 'MODELO BASE'} | {len(teste)} tarefas\n")

# FUNÇÕES 
def gerar_saida(tarefa):
    """Monta o prompt no formato ChatML do Qwen e gera a resposta."""
    
    # O Qwen-3B-Instruct já é inteligente o suficiente para não precisar de 1-Shot,
    # um bom System Prompt já o força a gerar JSON.
    prompt = f"""<|im_start|>system
Você é um extrator de entidades JSON. Responda APENAS com o JSON.<|im_end|>
<|im_start|>user
{tarefa}<|im_end|>
<|im_start|>assistant
"""
    entrada = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        saida = model.generate(
            **entrada, 
            max_new_tokens=MAX_TOKENS, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    tamanho_prompt = entrada["input_ids"].shape[1]
    novos = saida[0][tamanho_prompt:]
    
    return tokenizer.decode(novos, skip_special_tokens=True).strip()

def extrair_json_seguro(texto):
    """Remove qualquer tag Markdown em volta do JSON."""
    match = re.search(r'\{.*?\}', texto, re.DOTALL)
    if match:
        return match.group(0)
    return texto 

def testa_extracao(gerado, esperado_str):
    texto_limpo = extrair_json_seguro(gerado)
    
    if not texto_limpo:
        return False, "Falha: O modelo não gerou um JSON."
    
    try:
        resultado = json.loads(texto_limpo)
        gabarito = json.loads(esperado_str)
        erros = []
        
        if gabarito.get("resultado") == "não consta":
            if resultado.get("resultado") != "não consta":
                return False, f"Falha: Esperava recusa ('não consta'), gerou {resultado}"
            return True, ""
            
        chaves_obrigatorias = ["local", "crime", "arma", "tipo_arma", "estado_vitima", "fonte"]
        for chave in chaves_obrigatorias:
            val_gerado = resultado.get(chave)
            val_esperado = gabarito.get(chave)
            if val_gerado != val_esperado:
                erros.append(f"[{chave}: esp={val_esperado!r} | ger={val_gerado!r}]")
                
        if erros:
            return False, " | ".join(erros)
            
        return True, ""
    except json.JSONDecodeError:
        return False, "Erro crítico: JSON quebrado."

# AVALIAÇÃO
resultados = []
inicio = time.time()
for i, ex in enumerate(teste, 1):
    texto_gerado = gerar_saida(ex["tarefa"])
    ok, erro = testa_extracao(texto_gerado, ex["codigo"])
    
    json_resumo = extrair_json_seguro(texto_gerado)
    resumo_tela = json_resumo[:55].replace('\n', '') + ("..." if len(json_resumo) > 55 else "")
    
    resultados.append({**ex, "gerado": texto_gerado, "passou": ok, "erro": erro})
    print(f"[{i:2d}/{len(teste)}] {'PASSOU' if ok else 'falhou'} "
          f"gerou: {resumo_tela!r}  {erro}")

# RESUMO 
print(f"\nTempo: {time.time() - inicio:.0f} s")
print("=" * 50)
for tipo in ["variacao", "inedita"]:
    grupo = [r for r in resultados if r.get("tipo") == tipo]
    if grupo:
        acertos = sum(r["passou"] for r in grupo)
        print(f"{tipo:<10}: {acertos:2d}/{len(grupo)}  ({acertos / len(grupo):.0%})")
total = sum(r["passou"] for r in resultados)
print(f"{'TOTAL':<10}: {total:2d}/{len(resultados)}  ({total / len(resultados):.0%})")
print("=" * 50)

# Mostra 2 exemplos para debug
for r in resultados[:2]:
    print(f"\nTAREFA: {r['tarefa']}")
    print(f"--- esperado ---\n{r['codigo']}")
    print(f"--- gerado ({'passou' if r['passou'] else 'falhou'}) ---\n{r['gerado']}")

# SALVAR RESULTADOS 
Path("resultados").mkdir(exist_ok=True)
nome = "lora-treinado" if USAR_MODELO_TREINADO else "qwen-base"
with open(f"resultados/{nome}.json", "w", encoding="utf-8") as arq:
    json.dump(resultados, arq, ensure_ascii=False, indent=2)
print(f"\nResultados salvos em resultados/{nome}.json")

Hardware: cuda
Configurando Quantização 4-bits...
Carregando Tokenizador e Modelo Base (Qwen/Qwen2.5-3B-Instruct)...


Loading weights: 100%|██████████| 434/434 [00:08<00:00, 53.58it/s]


Injetando o Adaptador LoRA (modelo-lora-rag)...

Avaliando MODELO TREINADO (LoRA) | 30 tarefas

[ 1/30] falhou gerou: '{"local": "rua augusta", "crime": "agressão", "arma": t...'  [arma: esp=None | ger=True]
[ 2/30] falhou gerou: '{"local": "estacionamento do shopping", "crime": "ameaç...'  [tipo_arma: esp='desconhecido' | ger='desarmado']
[ 3/30] PASSOU gerou: '{"local": "passarela", "crime": "vandalismo", "arma": t...'  
[ 4/30] falhou gerou: '{"local": "estação da luz", "crime": "ameaça", "arma": ...'  [arma: esp=None | ger=False]
[ 5/30] PASSOU gerou: '{"local": "rua de trás", "crime": "agressão", "arma": t...'  
[ 6/30] falhou gerou: '{"localizacao": "não consta", "texto": "nenhum document...'  Falha: Esperava recusa ('não consta'), gerou {'localizacao': 'não consta', 'texto': 'nenhum documento relevante encontrado'}
[ 7/30] PASSOU gerou: '{"local": "desconhecido", "crime": "furto", "arma": tru...'  
[ 8/30] PASSOU gerou: '{"local": "rua augusta", "crime": "assalto", "arma": fa...